### Provider Master Prep Workflow -> Part I
#### Creating a reusable master provider file to clean provider names
##### Part-1 is to identify mismatches in HOC names between two sources and fixing the missing data, identifying the level of data for optimal match

In [1]:
### Importing required libraries
import pandas as pd

In [2]:
### Reading the providers CSV file
providers_data = pd.read_csv('../data/bdc_us_provider_list_J25_03mar2026.csv')
print(providers_data.shape)
providers_data.head()


(2883, 3)


,frn,provider_id,holding_company
0,12609,240058,Mid-Hudson Cablevision
1,12781,240068,"Neu Ventures, Inc."
2,13722,131087,"Rainbow Telecommunications Association, Inc."
3,14936,240058,Mid-Hudson Cablevision
4,17640,131060,"TPC ACC Acquisition, LLC"


In [3]:
### Printing out unique number of provider ids
unique_provider_ids = providers_data['provider_id'].nunique()
print(f'Unique provider IDs: {unique_provider_ids}')

Unique provider IDs: 2179


In [4]:
### Checking the level of the data by grouping on provider_id and counting the number of records for each provider
provider_counts = providers_data.groupby('provider_id').size()

### Printing multiple provider id rows
multiple_provider_ids = providers_data[providers_data['provider_id'].isin(provider_counts[provider_counts > 1].index)]
print(multiple_provider_ids.head())

       frn  provider_id                               holding_company
0    12609       240058                        Mid-Hudson Cablevision
2    13722       131087  Rainbow Telecommunications Association, Inc.
3    14936       240058                        Mid-Hudson Cablevision
4    17640       131060                      TPC ACC Acquisition, LLC
7  1514090       131161                     Waimana Enterprises, Inc.


In [5]:
### Checking whether provider id + holding_company combination is unique
providers_without_frn = providers_data.drop(columns=['frn'])
unique_combinations = providers_without_frn.drop_duplicates(subset=['provider_id', 'holding_company'])
print(f'Unique provider_id + holding_company combinations: {len(unique_combinations)}')

Unique provider_id + holding_company combinations: 2179


> This proves that FRN number was different for different providers with the same holding company

In [6]:
### Reading provider holding company XLSX sheet
providers_holding_company = pd.read_excel('../data/BDC Provider ID Table of Service Providers.xlsx')
print(providers_holding_company.shape)
providers_holding_company.head()

(4456, 5)


,Provider Name,Holding Company,Operation Type,FRN,Provider ID
0,"@Link Services, LLC","AtLink Services, LLC",Non-ILEC,16085920,290004
1,1 Point Communications,1 Point Communications,Non-ILEC,21352968,270002
2,101Netlink,101Netlink,Non-ILEC,18247254,190002
3,"123.Net, Inc","123.Net, Inc.",Non-ILEC,8590846,460000
4,"1Path Managed Services, LLC","1Path Managed Services, LLC",Non-ILEC,34347336,490000


In [7]:
### Printing out unique number of provider ids in the holding company data
unique_provider_ids_holding_company = providers_holding_company['Provider ID'].nunique()
print(f'Unique provider IDs in holding company data: {unique_provider_ids_holding_company}')

Unique provider IDs in holding company data: 3619


In [8]:
### Removing the Provider and FRN columns and checking if the remaining columns are unique
providers_holding_company_without_id_frn = providers_holding_company.drop(columns=['Provider Name', 'FRN', 'Operation Type'])
unique_combinations_holding_company = providers_holding_company_without_id_frn.drop_duplicates()
print(f'Unique combinations of remaining columns: {len(unique_combinations_holding_company)}')

Unique combinations of remaining columns: 3619


> This proves that Provider Name and FRN are linked together to distinguish sister corporations under the same holding company umbrella

In [9]:
### Renaming columns for merging
providers_data.rename(columns={'provider_id': 'Provider ID', 'frn': 'FRN'}, inplace=True)

In [10]:
### Merging the two datasets on the 'Provider ID' column
merged_data = pd.merge(providers_data, providers_holding_company, on=['Provider ID', 'FRN'], how='left')
### Displaying the merged dataset
print(merged_data.shape)
merged_data.head()

(2883, 6)


,FRN,Provider ID,holding_company,Provider Name,Holding Company,Operation Type
0,12609,240058,Mid-Hudson Cablevision,CATSKILL MOUNTAIN CABLEVISION,Mid-Hudson Cablevision,Non-ILEC
1,12781,240068,"Neu Ventures, Inc.","NEU VENTURES, INC.","Neu Ventures, Inc.",Non-ILEC
2,13722,131087,"Rainbow Telecommunications Association, Inc.",RAINBOW COMMUNICATIONS LLC,"Rainbow Telecommunications Association, Inc.",Non-ILEC
3,14936,240058,Mid-Hudson Cablevision,"MID-HUDSON CABLEVISION, Inc.",Mid-Hudson Cablevision,Non-ILEC
4,17640,131060,"TPC ACC Acquisition, LLC","Blue Stream Communications, LLC","TPC ACC Acquisition, LLC",Non-ILEC


In [11]:
### Finding rows where the 'holding_company' and 'Holding Company' columns do not match
mismatched_rows = merged_data[merged_data['holding_company'] != merged_data['Holding Company']]
### Displaying the mismatched rows
print(mismatched_rows.shape)
mismatched_rows

(225, 6)


,FRN,Provider ID,holding_company,Provider Name,Holding Company,Operation Type
38,1581297,130335,"Fidium Holdings, LLC",NaN,NaN,NaN
41,1583962,520000,"Okanogan County Electric Cooperative, Inc.",NaN,NaN,NaN
44,1584960,130029,Alaska Power & Telephone Company,Alaska Telephone Company,"Alaska Power & Telephone, Inc.",ILEC
51,1598598,520001,"City of Salem, Utah",NaN,NaN,NaN
110,1687813,330025,Uniti Group Inc.,NaN,NaN,NaN
...,...,...,...,...,...,...
2878,37070679,130045,"Alpine Communications, LLC",NaN,NaN,NaN
2879,37202561,520066,"Logan County Gig, LLC",NaN,NaN,NaN
2880,37365616,520068,"Tammany Wireless Holdings, LLC",NaN,NaN,NaN
2881,37397841,520069,"Wave 7 Communications, LLC",NaN,NaN,NaN


In [12]:
### Printing out the mismatched rows where the 'holding_company' and 'Holding Company' columns are both not null
mismatched_non_null_rows = mismatched_rows[(mismatched_rows['holding_company'].notnull()) & (mismatched_rows['Holding Company'].notnull())]
print(mismatched_non_null_rows.shape)
mismatched_non_null_rows

(63, 6)


,FRN,Provider ID,holding_company,Provider Name,Holding Company,Operation Type
44,1584960,130029,Alaska Power & Telephone Company,Alaska Telephone Company,"Alaska Power & Telephone, Inc.",ILEC
159,1772888,130068,West Kentucky Rural Telephone Cooperative Corp...,Ardmore Telephone Company Inc,"Synergy Technology Partners, inc.",ILEC
163,1775295,131378,"Future Fiber FinCo, LLC",Brindlee Mountain Telephone LLC,Otelco Inc.,ILEC
174,1792621,130068,West Kentucky Rural Telephone Cooperative Corp...,West Kentucky Rural Telephone Cooperative Corp...,"Synergy Technology Partners, inc.",ILEC
424,3007481,130619,The Chillicothe Telephone Co,The Chillicothe Telephone Co,"Horizon Telcom, Inc.",ILEC
...,...,...,...,...,...,...
2775,33307901,470030,Boston Omaha Corporation,"Fiber Fast Homes, LLC",Boston Omaha,Non-ILEC
2786,33445198,131378,"Future Fiber FinCo, LLC","Netspeed Management, Inc.",Otelco Inc.,Non-ILEC
2792,33590407,340068,Skynet Country Inc.,Skynet Country Inc,"Skynet Country, LLC",Non-ILEC
2802,33889254,300191,"Trailblazer Holdco, LLC","Lumos Fiber of South Carolina, LLC",Gridiron Fiber Corporation,Non-ILEC


In [13]:
### Printing out the mismatched rows where the 'holding_company' column is null but the 'Holding Company' column is not null
mismatched_holding_company_not_null = mismatched_rows[(mismatched_rows['holding_company'].isnull()) & (mismatched_rows['Holding Company'].notnull())]
print(mismatched_holding_company_not_null.shape)

(0, 6)


In [14]:
### Printing out the mismatched rows where the 'holding_company' column is not null but the 'Holding Company' column is null
mismatched_holding_company_null = mismatched_rows[(mismatched_rows['holding_company'].notnull()) & (mismatched_rows['Holding Company'].isnull())]
print(mismatched_holding_company_null.shape)

(162, 6)


In [15]:
### Printing out rows where both 'holding_company' and 'Holding Company' columns are null
both_holding_company_null = merged_data[(merged_data['holding_company'].isnull()) & (merged_data['Holding Company'].isnull())]
print(both_holding_company_null.shape)

(0, 6)


* These mismatches need to be manually checked and fixed, in other cases if one HOC name is absent / null we can fill it with the other HOC name (162 such cases)

In [16]:
### Exporting the mismatched rows to a CSV file for further analysis
mismatched_non_null_rows.to_csv('../output/mismatched_provider_holding_company.csv', index=False)